# Lab 8A: First-Order Logic Models, Symbols, and Interpretations

## Learning goals
By the end of this lab, students can:
- explain how first-order logic extends propositional logic with **objects**, **relations**, and **functions**;
- distinguish a symbol from its **interpretation** in a model;
- evaluate atomic, complex, universal, and existential first-order sentences over a finite model;
- visually connect Chapter 8's Richard/John model to executable Python structures.

## Chapter 8 connection
Chapter 8 says that first-order logic is useful because it can represent many objects with relations among them. A model contains a domain of objects and an interpretation mapping constants to objects, predicates to relations, and functions to functions. This notebook builds a small model inspired by the chapter's king/brother/crown example and evaluates formulas against it.


In [ ]:
# Run this cell first.
# Required packages: matplotlib. ipywidgets is optional for dropdown controls.

from dataclasses import dataclass
from itertools import product
from html import escape
import math
import matplotlib.pyplot as plt
from IPython.display import display, HTML, clear_output

try:
    import ipywidgets as widgets
    WIDGETS_AVAILABLE = True
except Exception:
    WIDGETS_AVAILABLE = False
    print("ipywidgets is not installed. The lab still works; interactive dropdowns will be skipped.")


## 1. A tiny first-order model

The **domain** contains objects. A model also gives meanings to the symbols we write in formulas.

In this model:
- constant symbols such as `Richard` and `John` name objects;
- predicate symbols such as `Brother`, `Person`, `King`, `Crown`, and `OnHead` name relations;
- the function symbol `LeftLeg` maps a person to the person's left-leg object.


In [ ]:
@dataclass
class FOLModel:
    domain: tuple
    constants: dict
    predicates: dict
    functions: dict

    def relation(self, name):
        return self.predicates.get(name, set())

    def function(self, name):
        return self.functions.get(name, {})

DOMAIN = (
    "Richard_obj",
    "John_obj",
    "Richard_left_leg",
    "John_left_leg",
    "Crown_obj",
)

model = FOLModel(
    domain=DOMAIN,
    constants={
        "Richard": "Richard_obj",
        "John": "John_obj",
        "CrownObject": "Crown_obj",
    },
    predicates={
        "Brother": {("Richard_obj", "John_obj"), ("John_obj", "Richard_obj")},
        "OnHead": {("Crown_obj", "John_obj")},
        "Person": {("Richard_obj",), ("John_obj",)},
        "King": {("John_obj",)},
        "Crown": {("Crown_obj",)},
    },
    functions={
        "LeftLeg": {
            ("Richard_obj",): "Richard_left_leg",
            ("John_obj",): "John_left_leg",
            # In strict FOL, functions are total. This demonstration keeps only useful mappings visible.
            ("Richard_left_leg",): "Richard_left_leg",
            ("John_left_leg",): "John_left_leg",
            ("Crown_obj",): "Crown_obj",
        }
    },
)

NICE = {
    "Richard_obj": "Richard",
    "John_obj": "John",
    "Richard_left_leg": "Richard's left leg",
    "John_left_leg": "John's left leg",
    "Crown_obj": "crown",
}


def nice(obj):
    return NICE.get(obj, str(obj))


In [ ]:
def draw_model(m=model, title="A first-order model: objects, relations, and a function"):
    pos = {
        "Richard_obj": (0.15, 0.55),
        "John_obj": (0.75, 0.55),
        "Richard_left_leg": (0.22, 0.15),
        "John_left_leg": (0.80, 0.15),
        "Crown_obj": (0.75, 0.92),
    }
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.set_title(title)
    ax.axis("off")

    # Draw objects and unary predicate labels.
    for obj, (x, y) in pos.items():
        circle = plt.Circle((x, y), 0.075, fill=False, linewidth=2)
        ax.add_patch(circle)
        labels = [p for p, tuples in m.predicates.items() if (obj,) in tuples and p not in {"OnHead", "Brother"}]
        ax.text(x, y, nice(obj), ha="center", va="center", fontsize=10)
        if labels:
            ax.text(x, y - 0.12, ", ".join(labels), ha="center", va="top", fontsize=9)

    def arrow(a, b, label, rad=0.0, style="->", linestyle="-"):
        ax.annotate(
            "",
            xy=pos[b], xytext=pos[a],
            arrowprops=dict(arrowstyle=style, lw=1.8, linestyle=linestyle,
                            connectionstyle=f"arc3,rad={rad}")
        )
        mx = (pos[a][0] + pos[b][0]) / 2
        my = (pos[a][1] + pos[b][1]) / 2
        ax.text(mx, my + 0.04 * (1 if rad >= 0 else -1), label, ha="center", fontsize=9)

    arrow("Richard_obj", "John_obj", "Brother", rad=0.25)
    arrow("John_obj", "Richard_obj", "Brother", rad=0.25)
    arrow("Crown_obj", "John_obj", "OnHead", rad=0.0)
    arrow("Richard_obj", "Richard_left_leg", "LeftLeg", linestyle="--")
    arrow("John_obj", "John_left_leg", "LeftLeg", linestyle="--")

    ax.text(0.02, 0.98, "Domain objects are circles.\nUnary predicates are labels.\nBinary relations/functions are arrows.",
            ha="left", va="top", fontsize=10)
    plt.show()


draw_model()


## 2. A small evaluator for first-order formulas

This evaluator is intentionally small and finite-domain. It is not a full theorem prover. It is a semantic evaluator: it checks whether a sentence is true in this particular model.


In [ ]:
class Expr:
    pass

@dataclass(frozen=True)
class Const(Expr):
    name: str
    def __str__(self): return self.name

@dataclass(frozen=True)
class Var(Expr):
    name: str
    def __str__(self): return self.name

@dataclass(frozen=True)
class Func(Expr):
    name: str
    args: tuple
    def __str__(self): return f"{self.name}(" + ", ".join(map(str, self.args)) + ")"

@dataclass(frozen=True)
class Atom(Expr):
    pred: str
    args: tuple
    def __str__(self): return f"{self.pred}(" + ", ".join(map(str, self.args)) + ")"

@dataclass(frozen=True)
class Eq(Expr):
    left: Expr
    right: Expr
    def __str__(self): return f"({self.left} = {self.right})"

@dataclass(frozen=True)
class Not(Expr):
    f: Expr
    def __str__(self): return f"¬{self.f}"

@dataclass(frozen=True)
class And(Expr):
    left: Expr
    right: Expr
    def __str__(self): return f"({self.left} ∧ {self.right})"

@dataclass(frozen=True)
class Or(Expr):
    left: Expr
    right: Expr
    def __str__(self): return f"({self.left} ∨ {self.right})"

@dataclass(frozen=True)
class Implies(Expr):
    left: Expr
    right: Expr
    def __str__(self): return f"({self.left} ⇒ {self.right})"

@dataclass(frozen=True)
class Iff(Expr):
    left: Expr
    right: Expr
    def __str__(self): return f"({self.left} ⇔ {self.right})"

@dataclass(frozen=True)
class ForAll(Expr):
    var: str
    body: Expr
    def __str__(self): return f"∀{self.var} {self.body}"

@dataclass(frozen=True)
class Exists(Expr):
    var: str
    body: Expr
    def __str__(self): return f"∃{self.var} {self.body}"


def eval_term(term, m, env):
    if isinstance(term, Const):
        return m.constants[term.name]
    if isinstance(term, Var):
        return env[term.name]
    if isinstance(term, Func):
        args = tuple(eval_term(a, m, env) for a in term.args)
        table = m.function(term.name)
        if args not in table:
            raise KeyError(f"Function {term.name}{args} is undefined in this demonstration model.")
        return table[args]
    raise TypeError(term)


def eval_formula(formula, m, env=None, trace=None, depth=0):
    env = {} if env is None else dict(env)
    trace = [] if trace is None else trace
    indent = "  " * depth

    if isinstance(formula, Atom):
        args = tuple(eval_term(a, m, env) for a in formula.args)
        value = args in m.relation(formula.pred)
        trace.append(f"{indent}{formula}: arguments {tuple(nice(a) for a in args)} -> {value}")
        return value
    if isinstance(formula, Eq):
        left, right = eval_term(formula.left, m, env), eval_term(formula.right, m, env)
        value = left == right
        trace.append(f"{indent}{formula}: {nice(left)} == {nice(right)} -> {value}")
        return value
    if isinstance(formula, Not):
        value = not eval_formula(formula.f, m, env, trace, depth + 1)
        trace.append(f"{indent}{formula} -> {value}")
        return value
    if isinstance(formula, And):
        left = eval_formula(formula.left, m, env, trace, depth + 1)
        right = eval_formula(formula.right, m, env, trace, depth + 1)
        value = left and right
        trace.append(f"{indent}{formula} -> {value}")
        return value
    if isinstance(formula, Or):
        left = eval_formula(formula.left, m, env, trace, depth + 1)
        right = eval_formula(formula.right, m, env, trace, depth + 1)
        value = left or right
        trace.append(f"{indent}{formula} -> {value}")
        return value
    if isinstance(formula, Implies):
        left = eval_formula(formula.left, m, env, trace, depth + 1)
        right = eval_formula(formula.right, m, env, trace, depth + 1)
        value = (not left) or right
        trace.append(f"{indent}{formula} -> {value}")
        return value
    if isinstance(formula, Iff):
        left = eval_formula(formula.left, m, env, trace, depth + 1)
        right = eval_formula(formula.right, m, env, trace, depth + 1)
        value = left == right
        trace.append(f"{indent}{formula} -> {value}")
        return value
    if isinstance(formula, ForAll):
        results = []
        for obj in m.domain:
            env[formula.var] = obj
            result = eval_formula(formula.body, m, env, trace, depth + 1)
            results.append(result)
            trace.append(f"{indent}  with {formula.var}={nice(obj)}: {result}")
        value = all(results)
        trace.append(f"{indent}{formula} -> {value}")
        return value
    if isinstance(formula, Exists):
        results = []
        for obj in m.domain:
            env[formula.var] = obj
            result = eval_formula(formula.body, m, env, trace, depth + 1)
            results.append(result)
            trace.append(f"{indent}  with {formula.var}={nice(obj)}: {result}")
        value = any(results)
        trace.append(f"{indent}{formula} -> {value}")
        return value
    raise TypeError(formula)

# Convenience constructors.
Richard = Const("Richard")
John = Const("John")
CrownObject = Const("CrownObject")
x = Var("x")
y = Var("y")

def P(name, *args): return Atom(name, tuple(args))
def LeftLeg(t): return Func("LeftLeg", (t,))


In [ ]:
FORMULAS = {
    "Brother(Richard, John)": P("Brother", Richard, John),
    "Brother(LeftLeg(Richard), John)": P("Brother", LeftLeg(Richard), John),
    "Crown(CrownObject) ∧ OnHead(CrownObject, John)": And(P("Crown", CrownObject), P("OnHead", CrownObject, John)),
    "∀x King(x) ⇒ Person(x)": ForAll("x", Implies(P("King", x), P("Person", x))),
    "∃x Crown(x) ∧ OnHead(x, John)": Exists("x", And(P("Crown", x), P("OnHead", x, John))),
    "∀x Person(x) ⇒ ∃y Brother(x,y)": ForAll("x", Implies(P("Person", x), Exists("y", P("Brother", x, y)))),
    "∃x King(x) ∧ Brother(Richard,x)": Exists("x", And(P("King", x), P("Brother", Richard, x))),
}

def evaluate_formula_by_name(name):
    formula = FORMULAS[name]
    trace = []
    value = eval_formula(formula, model, trace=trace)
    html = f"<h3>{escape(str(formula))}</h3><p><b>Truth value in this model:</b> {value}</p>"
    html += "<pre style='background:#f6f8fa;padding:10px;border:1px solid #ddd;'>" + escape("\n".join(trace[-40:])) + "</pre>"
    display(HTML(html))

for name in FORMULAS:
    trace = []
    value = eval_formula(FORMULAS[name], model, trace=trace)
    print(f"{name:52s} -> {value}")


In [ ]:
if WIDGETS_AVAILABLE:
    dropdown = widgets.Dropdown(options=list(FORMULAS.keys()), description="Formula:", layout=widgets.Layout(width="80%"))
    out = widgets.Output()

    def on_change(change=None):
        with out:
            clear_output(wait=True)
            draw_model(title="Model used for the selected formula")
            evaluate_formula_by_name(dropdown.value)

    dropdown.observe(on_change, names="value")
    display(dropdown, out)
    on_change()
else:
    draw_model(title="Model used for a sample formula")
    evaluate_formula_by_name("∀x King(x) ⇒ Person(x)")


## 3. Student practice

1. Add `King(Richard)` to the predicate table and evaluate all formulas again. Which values change?
2. Change the interpretation of the constant `Richard` so that it points to `Crown_obj`. What happens to `Brother(Richard, John)`?
3. Write a new formula for: "Some person has a left leg." Should it be true in this model?

This is the central Chapter 8 lesson: the same sentence can change truth value when its model or interpretation changes.
